# 🏥 Multimodal AI Benchmark on ROP-VL Dataset
### Vision-Language & Multimodal Systems in Clinical Settings
**Institution:** FAST-NUCES, Karachi Campus | **Lab:** GUTECH On-Premises GPU Lab Transition  
**Team:** Ibad Ur Rehman, Muhammad Mustafa, Sahil Latif

---
### Complete Modular Pipeline Structure:
1. **Environment Setup & Packages**
2. **Dataset Acquisition & Extraction (ROP-VL)**
3. **Preprocessing Stream 1: Vision Modality (Fundus Images & Augmentation)**
4. **Preprocessing Stream 2: Language Modality (Clinical Text & BioClinicalBERT Tokenization)**
5. **Preprocessing Stream 3: Tabular Modality (Biological Metadata Normalization)**
6. **Preprocessing Stream 4: Patient-Level 5-Fold Splitting (Zero Data Leakage)**
7. **Unified 3-Modality PyTorch Dataset & DataLoader**
8. **Modality Encoders (ResNet-50, BioClinicalBERT, 3-Layer MLP)**
9. **Fusion Architecture 1: Early Fusion (Feature Concatenation)**
10. **Fusion Architecture 2: Late Fusion (Decision Averaging)**
11. **Fusion Architecture 3: Intermediate Fusion (Relational Cross-Attention / GAT)**
12. **Training & Validation Engine (AdamW + Cosine Annealing)**
13. **Model Execution: Training All 3 Fusion Strategies**
14. **Comparative Evaluation Benchmark (Accuracy, Macro F1, Macro AUC)**
15. **Interactive Single-Patient Inference & Cross-Modal Explainability Heatmap**

--- 
## ⚙️ Cell 1: Environment Setup & GPU Verification

In [ ]:
!nvidia-smi
!pip install -q transformers timm openpyxl seaborn scikit-learn tabulate matplotlib torchvision

--- 
## 📥 Cell 2: Dataset Acquisition & Extraction (ROP-VL from Figshare)

In [ ]:
import os
import zipfile
import urllib.request

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
zip_path = os.path.join(DATA_DIR, "ROP-VL.zip")
download_url = "https://ndownloader.figshare.com/files/58329352"

if not os.path.exists(os.path.join(DATA_DIR, "img_info.xlsx")):
    if not os.path.exists(zip_path):
        print("[*] Downloading ROP-VL.zip (~656 MB) from official Figshare repository...")
        urllib.request.urlretrieve(download_url, zip_path)
        print("[+] Download complete!")
    print("[*] Extracting dataset files...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
    print("[+] Extraction completed successfully!")
else:
    print("[+] ROP-VL dataset already extracted and verified in:", DATA_DIR)

!ls -lh data

--- 
## 👁️ Cell 3: Preprocessing Stream 1 — Vision Modality (Fundus Transforms & Augmentation)
- Resizes color fundus retinal photographs to $224 \times 224$.
- Applies clinical data augmentations (random rotation, horizontal flip, color jitter) to simulate varied camera lighting and pupil dilation.
- Normalizes RGB channels using ImageNet stats.

In [ ]:
from torchvision import transforms

IMG_SIZE = 224

train_vision_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_vision_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("[+] Vision preprocessing pipelines (Train & Val) initialized.")

--- 
## 📝 Cell 4: Preprocessing Stream 2 — Language Modality (Expert Templates & Tokenization)
- Loads disease descriptions from `disease_description.xlsx`.
- Synthesizes the structured knowledge template $\pi_{\text{ROP-EK}}$ combining disease signs with patient context.
- Uses **BioClinicalBERT** tokenizer to produce `input_ids` and `attention_mask` (sequence length 96).

In [ ]:
from transformers import AutoTokenizer
import pandas as pd

# Load clinical descriptions per disease class
disease_desc_path = os.path.join(DATA_DIR, "disease_description.xlsx")
disease_descriptions = {}
if os.path.exists(disease_desc_path):
    desc_df = pd.read_excel(disease_desc_path)
    for _, row in desc_df.iterrows():
        disease_descriptions[str(row.iloc[0]).strip()] = str(row.iloc[1]).strip()

# Tokenizer setup (BioClinicalBERT with fallback to BERT-uncased)
tokenizer_name = "emilyalsentzer/Bio_ClinicalBERT"
try:
    text_tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
except Exception:
    text_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def build_clinical_text(category, gender, ga, bw, plurality, delivery_mode):
    desc = disease_descriptions.get(category, f"retinal findings associated with {category}")
    pl_str = "singleton" if plurality == 1 else "multiple birth"
    return f"A fundus photograph of {desc} in a {gender} infant born at {ga} weeks with birth weight of {bw}g from {pl_str} delivered via {delivery_mode}."

sample_text = build_clinical_text("Stage 2 ROP", "male", 28, 1100, 1, "cesarean section")
sample_tokens = text_tokenizer(sample_text, max_length=96, padding="max_length", truncation=True, return_tensors="pt")

print("[Sample Clinical Template]:", sample_text)
print("[Tokenized Shape]:", sample_tokens['input_ids'].shape)

--- 
## 📊 Cell 5: Preprocessing Stream 3 — Tabular Modality (Biological Metadata Normalization)
- 5 Patient Features: Gender, Gestational Age (GA), Birth Weight (BW), Plurality, Delivery Mode.
- Z-Score normalizes continuous physiological metrics using cohort statistics $(\mu_{GA}=30.40, \sigma_{GA}=3.85, \mu_{BW}=1490.37, \sigma_{BW}=732.03)$.
- Encodes categorical attributes into binary tensors.

In [ ]:
import torch

GA_MEAN, GA_STD = 30.40, 3.85
BW_MEAN, BW_STD = 1490.37, 732.03

def preprocess_tabular_features(gender, gestational_age, birth_weight, plurality, delivery_mode):
    # 1. Gender: Male=1.0, Female=0.0
    g_code = 1.0 if str(gender).lower() == "male" else 0.0
    # 2. Continuous Z-score scaling
    ga_norm = (float(gestational_age) - GA_MEAN) / GA_STD
    bw_norm = (float(birth_weight) - BW_MEAN) / BW_STD
    # 3. Plurality: Singleton=0.0, Twin/Multiple=1.0
    pl_code = float(plurality) - 1.0
    # 4. Delivery Mode: Vaginal=1.0, Cesarean=0.0
    dm_code = 1.0 if "vag" in str(delivery_mode).lower() else 0.0
    
    return torch.tensor([g_code, ga_norm, bw_norm, pl_code, dm_code], dtype=torch.float32)

sample_tab = preprocess_tabular_features("male", 28, 1100, 1, "cesarean section")
print("[Preprocessed Tabular Feature Vector (5-dim)]:", sample_tab)

--- 
## 🛡️ Cell 6: Preprocessing Stream 4 — Patient-Level 5-Fold Partitioning (Zero Data Leakage)
- Clusters images strictly by `patient_id`.
- Guarantees that multi-eye (bilateral) photos and follow-up examinations of the same infant never cross between train and validation splits.

In [ ]:
import numpy as np

img_info_path = os.path.join(DATA_DIR, "img_info.xlsx")
raw_df = pd.read_excel(img_info_path)

def create_patient_level_folds(df, total_folds=5, random_state=42):
    unique_patients = np.sort(df['patient_id'].unique())
    np.random.seed(random_state)
    shuffled_patients = np.random.permutation(unique_patients)
    folds = np.array_split(shuffled_patients, total_folds)
    return folds

folds = create_patient_level_folds(raw_df)
print(f"[+] Total Unique Patients: {len(raw_df['patient_id'].unique())} | Total Images: {len(raw_df)}")
for f_idx, fold_pts in enumerate(folds):
    print(f"    Fold {f_idx}: {len(fold_pts)} patients")

--- 
## 📦 Cell 7: Unified 3-Modality PyTorch Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

OCCURRENCE_4_CLASSES = {
    'Normal': 0,
    'Laser-treated ROP': 1,
    'Laser-treated': 1,
    'Stage 1 ROP': 2,
    'Stage 2 ROP': 2,
    'Stage 3 ROP': 2,
    'Stage 4 ROP': 2,
    'Stage 5 ROP': 2,
    'A-ROP': 3
}

class UnifiedROPDataset(Dataset):
    def __init__(self, df, data_dir, split="train", fold_val_patients=None):
        if fold_val_patients is not None:
            if split == "train":
                self.df = df[~df['patient_id'].isin(fold_val_patients)].reset_index(drop=True)
            else:
                self.df = df[df['patient_id'].isin(fold_val_patients)].reset_index(drop=True)
        else:
            self.df = df.copy()

        self.data_dir = data_dir
        self.split = split
        self.transforms = train_vision_transforms if split == "train" else val_vision_transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = str(row['image'])
        cat = str(row['categories']).strip()
        
        # 1. Vision Stream
        img_candidates = [
            os.path.join(self.data_dir, cat, img_name),
            os.path.join(self.data_dir, "images", cat, img_name),
            os.path.join(self.data_dir, img_name)
        ]
        img_path = next((p for p in img_candidates if os.path.exists(p)), None)
        if img_path:
            img = Image.open(img_path).convert('RGB')
        else:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), color=(130, 60, 40))
        img_t = self.transforms(img)

        # 2. Language Stream
        text = build_clinical_text(cat, row.get('gender', 'unknown'), row.get('gestational_age', 30),
                                   row.get('birth_weight', 1500), row.get('plurality', 1), row.get('delivery_mode', 'cesarean'))
        tokenized = text_tokenizer(text, max_length=96, padding="max_length", truncation=True, return_tensors="pt")

        # 3. Tabular Stream
        tab_t = preprocess_tabular_features(row.get('gender', 'unknown'), row.get('gestational_age', 30),
                                            row.get('birth_weight', 1500), row.get('plurality', 1), row.get('delivery_mode', 'cesarean'))

        # 4. Target
        label_t = torch.tensor(OCCURRENCE_4_CLASSES.get(cat, 0), dtype=torch.long)

        return {
            'image': img_t,
            'input_ids': tokenized['input_ids'].squeeze(0),
            'attention_mask': tokenized['attention_mask'].squeeze(0),
            'tabular': tab_t,
            'label': label_t
        }

val_patients_fold0 = set(folds[0])
train_dataset = UnifiedROPDataset(raw_df, DATA_DIR, split="train", fold_val_patients=val_patients_fold0)
val_dataset = UnifiedROPDataset(raw_df, DATA_DIR, split="val", fold_val_patients=val_patients_fold0)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

print(f"[+] Unified DataLoaders Ready! Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

--- 
## 🧠 Cell 8: Modality Encoders (Vision, Language, Tabular $\rightarrow$ Unified 256-D Space)
- `VisionEncoder`: ResNet-50 ImageNet backbone $\rightarrow$ Linear $[2048 \rightarrow 256]$
- `TextEncoder`: BioClinicalBERT backbone $\rightarrow$ Linear $[768 \rightarrow 256]$
- `TabularEncoder`: 3-Layer MLP $[5 \rightarrow 64 \rightarrow 128 \rightarrow 256]$

In [ ]:
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights
from transformers import AutoModel

class VisionEncoder(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.DEFAULT)
        in_f = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.proj = nn.Sequential(
            nn.Linear(in_f, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.proj(self.backbone(x))

class TextEncoder(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        try:
            self.bert = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
        except Exception:
            self.bert = AutoModel.from_pretrained("bert-base-uncased")
        for p in self.bert.parameters():
            p.requires_grad = False  # Freeze pretrained weights
        self.proj = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(inplace=True)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = out.last_hidden_state[:, 0, :]
        return self.proj(cls_token)

class TabularEncoder(nn.Module):
    def __init__(self, in_features=5, embed_dim=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Linear(128, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.mlp(x)

print("[+] Encoders defined: Vision(256-D), Text(256-D), Tabular(256-D).")

--- 
## 🔗 Cell 9: Fusion Architecture 1 — Early Fusion (Feature Concatenation)
- Flattens and concatenates features: $\mathbf{z} = [\mathbf{v} \,|\,|\, \mathbf{t} \,|\,|\, \mathbf{s}] \in \mathbb{R}^{768}$.

In [ ]:
class EarlyFusionModel(nn.Module):
    def __init__(self, num_classes=4, embed_dim=256):
        super().__init__()
        self.v_enc = VisionEncoder(embed_dim)
        self.t_enc = TextEncoder(embed_dim)
        self.s_enc = TabularEncoder(5, embed_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 3, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, batch):
        v = self.v_enc(batch['image'])
        t = self.t_enc(batch['input_ids'], batch['attention_mask'])
        s = self.s_enc(batch['tabular'])
        fused = torch.cat([v, t, s], dim=-1)
        return self.classifier(fused), {}

print("[+] Early Fusion architecture ready.")

--- 
## ⚖️ Cell 10: Fusion Architecture 2 — Late Fusion (Decision Averaging)
- Computes individual classification logits per modality.
- Ensembles predictions via learnable soft-voting weights: $\mathbf{l} = \sum w_m \mathbf{l}_m$.

In [ ]:
class LateFusionModel(nn.Module):
    def __init__(self, num_classes=4, embed_dim=256):
        super().__init__()
        self.v_enc = VisionEncoder(embed_dim)
        self.t_enc = TextEncoder(embed_dim)
        self.s_enc = TabularEncoder(5, embed_dim)
        self.v_head = nn.Linear(embed_dim, num_classes)
        self.t_head = nn.Linear(embed_dim, num_classes)
        self.s_head = nn.Linear(embed_dim, num_classes)
        self.weights = nn.Parameter(torch.ones(3) / 3.0)

    def forward(self, batch):
        v_logits = self.v_head(self.v_enc(batch['image']))
        t_logits = self.t_head(self.t_enc(batch['input_ids'], batch['attention_mask']))
        s_logits = self.s_head(self.s_enc(batch['tabular']))
        w = torch.softmax(self.weights, dim=0)
        final = w[0] * v_logits + w[1] * t_logits + w[2] * s_logits
        return final, {'weights': w.detach().cpu().numpy()}

print("[+] Late Fusion architecture ready.")

--- 
## 🕸️ Cell 11: Fusion Architecture 3 — Intermediate Fusion (Relational Cross-Attention / GAT)
- Treats modalities as graph nodes with learnable Modality Type Embeddings.
- Calculates dynamic message passing attention: $\alpha_{ij} = \text{Softmax}\left(\frac{Q_i K_j^T}{\sqrt{d}}\right)$.
- Extracts a $3 \times 3$ attention matrix providing transparent clinician explainability.

In [ ]:
class IntermediateFusionModel(nn.Module):
    def __init__(self, num_classes=4, embed_dim=256, num_heads=4):
        super().__init__()
        self.v_enc = VisionEncoder(embed_dim)
        self.t_enc = TextEncoder(embed_dim)
        self.s_enc = TabularEncoder(5, embed_dim)
        self.modality_embed = nn.Embedding(3, embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 3, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, batch):
        B = batch['image'].size(0)
        dev = batch['image'].device
        v = self.v_enc(batch['image'])
        t = self.t_enc(batch['input_ids'], batch['attention_mask'])
        s = self.s_enc(batch['tabular'])
        tokens = torch.stack([v, t, s], dim=1)
        mod_ids = torch.tensor([0, 1, 2], device=dev).unsqueeze(0).expand(B, -1)
        tokens = tokens + self.modality_embed(mod_ids)
        attn_out, attn_weights = self.attn(query=tokens, key=tokens, value=tokens)
        tokens = self.norm(tokens + attn_out)
        fused = tokens.reshape(B, -1)
        return self.classifier(fused), {'attention_matrix': attn_weights.detach().cpu()}

print("[+] Intermediate Cross-Attention / GAT Fusion architecture ready.")

--- 
## 🔄 Cell 12: Training & Validation Loop Engine (AdamW + Cosine Annealing)

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_and_eval(model, train_loader, val_loader, epochs=6, lr=1e-4, model_name="model"):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    best_f1 = 0.0

    print(f"\n---> Training {model_name.upper()} on {device} ({epochs} epochs) <---")
    for ep in range(1, epochs + 1):
        model.train()
        tr_loss, tr_preds, tr_labels = 0.0, [], []
        for batch in train_loader:
            for k in ['image', 'input_ids', 'attention_mask', 'tabular', 'label']:
                batch[k] = batch[k].to(device)
            optimizer.zero_grad()
            logits, _ = model(batch)
            loss = criterion(logits, batch['label'])
            loss.backward()
            optimizer.step()
            tr_loss += loss.item() * batch['label'].size(0)
            tr_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
            tr_labels.extend(batch['label'].cpu().numpy())
        scheduler.step()

        # Validation step
        model.eval()
        val_loss, val_preds, val_labels = 0.0, [], []
        with torch.no_grad():
            for batch in val_loader:
                for k in ['image', 'input_ids', 'attention_mask', 'tabular', 'label']:
                    batch[k] = batch[k].to(device)
                logits, _ = model(batch)
                loss = criterion(logits, batch['label'])
                val_loss += loss.item() * batch['label'].size(0)
                val_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
                val_labels.extend(batch['label'].cpu().numpy())

        tr_acc = accuracy_score(tr_labels, tr_preds)
        val_acc = accuracy_score(val_labels, val_preds)
        val_f1 = f1_score(val_labels, val_preds, average='macro', zero_division=0)
        print(f"Epoch [{ep:02d}/{epochs:02d}] Train Acc: {tr_acc*100:.1f}% || Val Acc: {val_acc*100:.1f}% | Val Macro F1: {val_f1:.4f}")
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), f"best_{model_name}.pth")
    return model

--- 
## 🚀 Cell 13: Model Execution — Training All 3 Fusion Strategies

In [ ]:
early_model = train_and_eval(EarlyFusionModel(num_classes=4), train_loader, val_loader, epochs=6, model_name="early")
late_model = train_and_eval(LateFusionModel(num_classes=4), train_loader, val_loader, epochs=6, model_name="late")
inter_model = train_and_eval(IntermediateFusionModel(num_classes=4), train_loader, val_loader, epochs=6, model_name="intermediate")

--- 
## 📈 Cell 14: Comparative Evaluation Benchmark & Plots

In [ ]:
from sklearn.metrics import roc_auc_score
from tabulate import tabulate
import seaborn as sns
import matplotlib.pyplot as plt

def benchmark_all_models(models_dict, val_loader):
    records = []
    for name, m in models_dict.items():
        m.eval()
        preds, labels, probs = [], [], []
        with torch.no_grad():
            for batch in val_loader:
                for k in ['image', 'input_ids', 'attention_mask', 'tabular', 'label']:
                    batch[k] = batch[k].to(device)
                logits, _ = m(batch)
                p = torch.softmax(logits, dim=-1).cpu().numpy()
                probs.extend(p)
                preds.extend(np.argmax(p, axis=-1))
                labels.extend(batch['label'].cpu().numpy())
        acc = accuracy_score(labels, preds)
        f1 = f1_score(labels, preds, average='macro', zero_division=0)
        try:
            auc = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
        except Exception:
            auc = 0.0
        records.append({
            'Fusion Paradigm': name,
            'Accuracy (%)': round(acc * 100, 2),
            'Macro F1': round(f1, 4),
            'Macro AUC': round(auc, 4)
        })
    return pd.DataFrame(records)

bench_df = benchmark_all_models({
    'Early Fusion (Concat)': early_model,
    'Late Fusion (Decision)': late_model,
    'Intermediate (Proposed Cross-Attn/GAT)': inter_model
}, val_loader)

print("\n" + tabulate(bench_df, headers='keys', tablefmt='psql', showindex=False))

# Comparison Visualization
plt.figure(figsize=(12, 4))
for i, col in enumerate(['Accuracy (%)', 'Macro F1', 'Macro AUC']):
    plt.subplot(1, 3, i+1)
    sns.barplot(x='Fusion Paradigm', y=col, data=bench_df, palette='viridis')
    plt.xticks(rotation=20, ha='right')
    plt.title(col, weight='bold')
plt.tight_layout()
plt.show()

--- 
## 🔬 Cell 15: Interactive Single-Patient Inference & Cross-Modal Explainability Heatmap

In [ ]:
sample_batch = next(iter(val_loader))
single_input = {k: v[:1].to(device) for k, v in sample_batch.items() if k in ['image', 'input_ids', 'attention_mask', 'tabular']}

inter_model.eval()
with torch.no_grad():
    logits, aux = inter_model(single_input)
    probs = torch.softmax(logits, dim=-1).squeeze(0).cpu().numpy()
    attn_mat = aux['attention_matrix'].squeeze(0).mean(dim=0).numpy()

class_names = ['Normal', 'Laser-treated ROP', 'Any Stage of ROP', 'Aggressive-ROP (A-ROP)']
pred_class = np.argmax(probs)

print(f"[*] PREDICTED DIAGNOSIS: ---> {class_names[pred_class]} <--- (Confidence: {probs[pred_class]*100:.2f}%)")

labels = ['Vision (Fundus)', 'Language (Notes)', 'Tabular (Metadata)']
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(attn_mat, annot=True, fmt=".2f", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title("Cross-Modal Attention Matrix (Explainability)", weight='bold')
plt.xlabel("Target Modality (Key)")
plt.ylabel("Source Modality (Query)")
plt.tight_layout()
plt.show()